# Post-Click Leakage Calculator

**One question:** given the leads you already paid for, which funnel stage destroys the most money?

This notebook decomposes a WhatsApp acquisition funnel into its five real stages and ranks each stage by **marginal economic impact** — the revenue unlocked per point of improvement, given current volume — *not* by lowest conversion rate.

> ⚠️ **No causality without real data.** This is a descriptive prioritization tool. It points to where money concentrates in a funnel; it does not prove that improving a stage *causes* a given revenue lift. Any scenario shown here is a **hypothesis to validate with a real before/after test**. See [`LIMITS_OF_USE.md`](../LIMITS_OF_USE.md).

All data in this notebook is **synthetic**. No real clients, no PII, no PHI.

In [1]:
import sys
sys.path.insert(0, "../src")
import csv
from leakage import FunnelInputs, compute, scenario, STAGE_LABELS, STAGE_HUMAN

# Load the synthetic dataset (generated by src/generate_synthetic.py, fixed seed)
with open("../data/synthetic_funnels.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"{len(rows)} synthetic storefronts loaded")
rows[0]

12 synthetic storefronts loaded


{'storefront': 'Tienda Aurora',
 'vertical': 'luxury_furniture',
 'monthly_ad_spend': '49328.0',
 'cpc': '11.64',
 'rate_message_per_click': '0.4722',
 'rate_reply_per_message': '0.7191',
 'rate_followup_per_reply': '0.2535',
 'rate_sale_per_followup': '0.2482',
 'avg_ticket': '61359.0',
 'gross_margin': '0.4838'}

## 1. The funnel decomposition

Revenue is the product of four stage rates times clicks times ticket:

$$\text{Revenue} = \text{clicks} \times p_1 \times p_2 \times p_3 \times p_4 \times \text{ticket}$$

where the stages are: message/click, reply/message, follow-up/reply, sale/follow-up.

In [2]:
def to_inputs(r):
    return FunnelInputs(
        monthly_ad_spend=float(r["monthly_ad_spend"]),
        cpc=float(r["cpc"]),
        rates={
            "message_per_click":  float(r["rate_message_per_click"]),
            "reply_per_message":  float(r["rate_reply_per_message"]),
            "followup_per_reply": float(r["rate_followup_per_reply"]),
            "sale_per_followup":  float(r["rate_sale_per_followup"]),
        },
        avg_ticket=float(r["avg_ticket"]),
        gross_margin=float(r["gross_margin"]),
    )

inp = to_inputs(rows[0])
res = compute(inp)

print(f"Storefront:  {rows[0]['storefront']} ({rows[0]['vertical']})")
print(f"Clicks:      {res.clicks:,.0f}")
print(f"Sales/mo:    {res.sales:,.1f}")
print(f"Revenue/mo:  ${res.revenue:,.0f}")
print(f"CAC:         ${res.cac:,.0f}")
print(f"ROAS:        {res.roas:.2f}x")

Storefront:  Tienda Aurora (luxury_furniture)
Clicks:      4,238
Sales/mo:    90.5
Revenue/mo:  $5,555,381
CAC:         $545
ROAS:        112.62x


## 2. The ranking that matters

The naive mistake is to attack the stage with the **lowest rate**. That is wrong: a terrible rate on a stage almost nobody reaches moves less money than a decent rate on a high-volume stage.

The correct prioritization ranks by the **economic derivative**:

$$\frac{\partial \text{Revenue}}{\partial p_i} = \frac{\text{Revenue}}{p_i}$$

A +1 percentage-point (absolute) improvement on stage $i$ unlocks $\frac{\text{Revenue}}{p_i}\times 0.01$ in incremental revenue. We rank stages by that quantity.

In [3]:
print("Marginal revenue per +1pp improvement (ranked):\n")
for k in res.ranking:
    flag = "  <-- PRIORITY" if k == res.priority_stage else ""
    print(f"  {STAGE_HUMAN[k]:24s}  ${res.marginal_gain[k]:>12,.0f}{flag}")

Marginal revenue per +1pp improvement (ranked):

  % sale / follow-up        $     223,827  <-- PRIORITY
  % follow-up / reply       $     219,147
  % message / click         $     117,649
  % reply / message         $      77,255


## 3. Scenario hypotheses (+10% / +20% / +30%)

We concentrate a *relative* uplift on the priority stage and read the incremental revenue.

**These are hypotheses, not forecasts.** The model assumes the uplift lands entirely on the priority stage and nothing else moves. Reality is messier — which is exactly why the deliverable to a client is a measured before/after, not this number.

In [4]:
for u in (0.10, 0.20, 0.30):
    s = scenario(inp, u)
    print(f"+{int(u*100)}% on {STAGE_HUMAN[s['stage']]}: "
          f"incremental ${s['incremental_revenue']:,.0f}/mo  "
          f"(ROAS {s['roas_before']:.1f}x -> {s['roas_after']:.1f}x)")

+10% on % sale / follow-up: incremental $555,538/mo  (ROAS 112.6x -> 123.9x)
+20% on % sale / follow-up: incremental $1,111,076/mo  (ROAS 112.6x -> 135.1x)
+30% on % sale / follow-up: incremental $1,666,614/mo  (ROAS 112.6x -> 146.4x)


## 4. Across the synthetic portfolio

The priority stage is not universal — it depends on each storefront's rate profile. That is the point: the tool finds *that business's* bottleneck, not a generic best practice.

In [5]:
print(f"{'Storefront':22s} {'Vertical':18s} {'Priority stage':22s} {'ROAS':>7s}")
print("-"*72)
for r in rows:
    rr = compute(to_inputs(r))
    print(f"{r['storefront']:22s} {r['vertical']:18s} "
          f"{STAGE_HUMAN[rr.priority_stage]:22s} {rr.roas:>6.1f}x")

Storefront             Vertical           Priority stage            ROAS
------------------------------------------------------------------------
Tienda Aurora          luxury_furniture   % sale / follow-up      112.6x
Casa Lumen             real_estate_lead   % sale / follow-up      130.2x
Estudio Roble          premium_decor      % sale / follow-up       52.7x
Marca Ficticia A       luxury_furniture   % follow-up / reply     161.2x
Comercio Demo B        real_estate_lead   % sale / follow-up      105.3x
Showroom Sintetico     premium_decor      % sale / follow-up       80.5x
Negocio Ejemplo C      luxury_furniture   % sale / follow-up      103.1x
Brand Placeholder D    real_estate_lead   % sale / follow-up      147.7x
Vertiente Ficticia     premium_decor      % sale / follow-up       55.5x
Demo Storefront E      luxury_furniture   % sale / follow-up      118.7x
Marca Prueba F         real_estate_lead   % sale / follow-up       83.5x
Cliente Sintetico G    premium_decor      % sale / 

## What this is — and what it is not

**Is:** a fast, auditable, descriptive prioritizer that turns a funnel into a ranked list of where money concentrates.

**Is not:** a causal model, a guarantee, or a substitute for measurement. The honest output to a client is always: *"with your current numbers, this stage has the highest economic leverage; we validate the hypothesis with a real before/after."*

See [`LIMITS_OF_USE.md`](../LIMITS_OF_USE.md) for the full boundary.